# Img2GPS — Walkthrough

CIS 5190 final project, Track A. We predict GPS coordinates from a single image taken on Penn's campus (test rectangle: 33rd & Walnut → 34th & Spruce). The official metric is the average Haversine distance in meters.

This notebook is the human-readable companion to the scripts:

- `Img2GPS/extract_exif.py` builds `metadata.csv`
- `Img2GPS/preprocess.py` provides `prepare_data` / `load_raw`
- `Img2GPS/model.py` defines the ResNet-18 regressor with target-normalization buffers
- `Img2GPS/train.py` runs the training loop
- `Img2GPS/eval_project_a.py` is the course-style evaluator


In [ ]:
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "Img2GPS").exists():
    REPO_ROOT = REPO_ROOT.parent
PROJECT_DIR = REPO_ROOT / "Img2GPS"
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(REPO_ROOT)

from preprocess import load_raw, prepare_data  # noqa: E402
from model import Model  # noqa: E402
from train import haversine_meters, location_grouped_split  # noqa: E402

REPO_ROOT, PROJECT_DIR

## 1. Data summary

We collected 89 photos around Penn's campus and extracted GPS coordinates from EXIF + Apple location xattrs. The location-grouped split makes sure photos that share an exact GPS coordinate (about 8 photos per spot) live entirely on one side of the train/val split, which avoids leakage.

In [ ]:
df = pd.read_csv(PROJECT_DIR / "metadata.csv")
print(f"rows: {len(df)}")
print(f"unique locations: {df[['latitude','longitude']].drop_duplicates().shape[0]}")
df.describe(percentiles=[0.05, 0.5, 0.95]).round(6)

In [ ]:
def haversine(a, b):
    R = 6_371_000.0
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

mean_lat = df['latitude'].mean()
mean_lon = df['longitude'].mean()
lat_span = haversine((df['latitude'].min(), mean_lon), (df['latitude'].max(), mean_lon))
lon_span = haversine((mean_lat, df['longitude'].min()), (mean_lat, df['longitude'].max()))
print(f"bounding box: {lat_span:.1f} m (NS) x {lon_span:.1f} m (EW)")
print(f"training mean: ({mean_lat:.6f}, {mean_lon:.6f})")

constant_dist = [haversine((mean_lat, mean_lon), p) for p in df[['latitude','longitude']].values]
print(f"constant-mean baseline Haversine: mean={np.mean(constant_dist):.2f}m  p50={np.median(constant_dist):.2f}m  max={np.max(constant_dist):.2f}m")

## 2. Train

Re-running the script keeps everything reproducible (seeded). Skip this cell if `model.pt` is already up to date.

In [ ]:
# !python Img2GPS/train.py --epochs 12 --lr 1e-3

## 3. Evaluate the saved checkpoint

We load `model.pt` (the best-by-Haversine snapshot) and report the official metrics on:

1. The full collected dataset.
2. The held-out validation split (location-grouped, the honest signal).
3. The course-provided reference set in `Img2GPS/reference/`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Model(weights_path=str(PROJECT_DIR / 'model.pt')).to(device).eval()
print(f"y_mean: {model.y_mean.tolist()}\ny_std:  {model.y_std.tolist()}")

In [ ]:
def evaluate(csv_path):
    X, y = prepare_data(str(csv_path))
    with torch.no_grad():
        preds = model(X.to(device)).cpu()
    distances = haversine_meters(preds, y).numpy()
    return preds, y, distances

preds_full, y_full, dists_full = evaluate(PROJECT_DIR / 'metadata.csv')
print(f"FULL set (n={len(y_full)}):")
print(f"  mean={dists_full.mean():.2f}m  p50={np.median(dists_full):.2f}m  p90={np.quantile(dists_full,0.9):.2f}m  max={dists_full.max():.2f}m")

In [ ]:
_, y_all = load_raw(str(PROJECT_DIR / 'metadata.csv'))
train_idx, val_idx = location_grouped_split(y_all, val_fraction=0.2, seed=42)
X_full, _ = prepare_data(str(PROJECT_DIR / 'metadata.csv'))
with torch.no_grad():
    preds_val = model(X_full[val_idx].to(device)).cpu()
dists_val = haversine_meters(preds_val, y_all[val_idx]).numpy()
print(f"VAL set (held-out, n={len(val_idx)}):")
print(f"  mean={dists_val.mean():.2f}m  p50={np.median(dists_val):.2f}m  p90={np.quantile(dists_val,0.9):.2f}m  max={dists_val.max():.2f}m")

In [ ]:
ref_csv = PROJECT_DIR / 'reference' / 'metadata.csv'
preds_ref, y_ref, dists_ref = evaluate(ref_csv)
print(f"REFERENCE set (n={len(y_ref)}):")
for (lat, lon), (plat, plon), d in zip(y_ref.tolist(), preds_ref.tolist(), dists_ref):
    print(f"  truth=({lat:.6f},{lon:.6f})  pred=({plat:.6f},{plon:.6f})  haversine={d:.2f}m")
print(f"reference mean Haversine: {dists_ref.mean():.2f}m")

## 4. Headline result

| Set | Mean Haversine (m) |
|---|---|
| Constant-mean baseline (training-set mean) | ~49 |
| Trained model — full collected set | see cell above |
| Trained model — held-out val | see cell above |
| Trained model — reference set | see cell above |

The held-out val number is the honest signal because the location-grouped split prevents the same exact location from appearing in both train and val.

## 5. Next iterations to consider

1. **Cover the full test rectangle.** Today the data hugs one corner of the spec's rectangle; the held-out val is in-distribution. Extra captures along Walnut, Spruce, and the cross streets would shrink the worst-case error on unseen test data.
2. **Stronger backbone (frozen)** like DINOv2 or CLIP-ViT, with a small MLP head — typically helps when data is small.
3. **Predict deltas from a fixed anchor point** instead of raw lat/lon. With target standardization this is largely redundant, but it pairs well with a stronger backbone.
4. **Heavier augmentation** (random shadows, blur, weather/lighting jitter) once we have more locations.